# BMED 712 — Track A | Week 3 Analysis Notebook

**Team:** Fatima Habib Farweh, Liang Li, Yasmine Khattab, Zehara Ali  
**Date:** 2026-04-06  
**Purpose:** Document all Week 3 work — bug analysis, feature validation, ML retraining, and professor feedback fixes.

---

## Table of Contents
1. [Setup & Imports](#1-setup)
2. [Feature Extraction Bug Analysis](#2-bug)
3. [New Feature Validation](#3-validation)
4. [ML Retraining — 3-Class](#4-3class)
5. [ML Retraining — 8-Class (Subtype)](#5-8class)
6. [Sensor Ablation](#6-ablation)
7. [Expanded Feature Experiment](#7-expanded)
8. [Professor Feedback Fixes — Figure 7 & Table II](#8-fixes)
9. [Summary & Key Insights](#9-summary)

<a id="1-setup"></a>
## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
from scipy.stats import kruskal, spearmanr
import warnings
warnings.filterwarnings("ignore")

# Paths
REPO = Path("..").resolve()
FREQ_DIR = REPO.parent / "frequency sheets"
RESULTS = REPO / "results"
ML_DIR = RESULTS / "ml_new_features"

# Plot style
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
})

META_COLS = {"subject_id", "trial_id", "window_idx", "label", "cohort", "phase", "win_s", "overlap"}
LABEL_COLORS = {"healthy": "#2ECC71", "neuro": "#E74C3C", "ortho": "#3498DB"}
COHORT_COLORS = {
    "HS": "#2ECC71", "RIL": "#C0392B", "PD": "#E74C3C",
    "CVA": "#E67E22", "CIPN": "#8E44AD",
    "KOA": "#2980B9", "HOA": "#1ABC9C", "ACL": "#F39C12",
}

print("Setup complete.")

<a id="2-bug"></a>
## 2. Feature Extraction Bug — Root Cause Analysis

During the 8-class analysis attempt, we discovered the original `master_features.csv` was generated with **three compounding errors**:

| Bug | Description | Impact |
|-----|-------------|--------|
| **Missing Acc channel** | Pipeline only extracted `FreeAcc` + `Gyr`, silently skipping raw `Acc` | Lost 1/3 of signals (168 vs 216 features) |
| **Trial-level aggregation** | Averaged all windows per trial into one row (1,356 rows) | Lost within-trial temporal variability |
| **No subtype label** | Only 3-class label (Healthy/Neuro/Ortho), no `cohort` column | 8-class analysis impossible |

Let's verify this by comparing the old and new feature sets:

In [ ]:
# Load old vs new features
old = pd.read_csv(REPO / "master_features.csv")
new = pd.read_csv(FREQ_DIR / "full_gait" / "features_win5000ms_ov50.csv")

feat_old = [c for c in old.columns if c not in {"trial_id", "subject_id", "label"}]
feat_new = [c for c in new.columns if c not in META_COLS]

# Signal types in each
sig_old = sorted(set(c.split("_")[1] for c in feat_old))
sig_new = sorted(set(c.split("_")[1] for c in feat_new))

comparison = pd.DataFrame({
    "Old (master_features.csv)": [
        f"{old.shape[0]:,}", f"{old.shape[1]}", f"{len(feat_old)}",
        ", ".join(sig_old), "Healthy/Neuro/Ortho", "No",
    ],
    "New (frequency sheets)": [
        f"{new.shape[0]:,}", f"{new.shape[1]}", f"{len(feat_new)}",
        ", ".join(sig_new), "healthy/neuro/ortho", "Yes (8 cohorts)",
    ],
}, index=["Rows", "Total columns", "Feature columns", "Signal types",
          "3-class label", "8-class cohort label"])

print("=== Old vs New Feature Comparison ===")
display(comparison)

In [ ]:
# Show exactly which channels are in old vs new
print("OLD feature channels (first 21 columns — HE sensor):")
he_old = [c for c in feat_old if c.startswith("HE_")][:21]
for c in he_old:
    print(f"  {c}")

print(f"\nNEW feature channels (HE sensor — all {len([c for c in feat_new if c.startswith('HE_')])} columns):")
he_new = [c for c in feat_new if c.startswith("HE_")]
for c in he_new:
    print(f"  {c}")

print(f"\n>>> Bug confirmed: OLD has NO 'Acc' channel — only FreeAcc and Gyr.")
print(f">>> NEW correctly includes Acc + FreeAcc + Gyr (18 features per sensor vs 14 before).")

<a id="3-validation"></a>
## 3. New Feature Validation

Fatemah re-extracted features from scratch. Let's validate the new dataset:
- Coverage: all phases, window sizes, and overlaps
- Class balance: 3-class and 8-class distributions
- Missing values
- Feature distributions (top discriminative features by Kruskal-Wallis)

In [ ]:
# Load ALL frequency-sheet CSVs
all_dfs = []
for phase_dir in sorted(FREQ_DIR.iterdir()):
    if not phase_dir.is_dir():
        continue
    for csv in sorted(phase_dir.glob("*.csv")):
        all_dfs.append(pd.read_csv(csv))

df_all = pd.concat(all_dfs, ignore_index=True)
feat_cols = [c for c in df_all.columns if c not in META_COLS]

print(f"Total windows loaded: {len(df_all):,}")
print(f"Phases: {sorted(df_all.phase.unique())}")
print(f"Features: {len(feat_cols)} sensor features + {len(META_COLS)} metadata columns")
print(f"Subjects: {df_all.subject_id.nunique()}")

In [ ]:
# 3a. Coverage table — rows per phase/window/overlap
coverage = (
    df_all.groupby(["phase", "win_s", "overlap"])
    .agg(n_windows=("label", "size"),
         n_subjects=("subject_id", "nunique"),
         healthy=("label", lambda x: (x == "healthy").sum()),
         neuro=("label", lambda x: (x == "neuro").sum()),
         ortho=("label", lambda x: (x == "ortho").sum()))
    .reset_index()
    .sort_values(["phase", "win_s", "overlap"])
)
print("=== Coverage Table (showing full_gait rows only) ===")
display(coverage[coverage.phase == "full_gait"].reset_index(drop=True))

In [ ]:
# 3b. Missing values
miss = df_all[feat_cols].isnull().mean()
miss_nonzero = miss[miss > 0].sort_values(ascending=False)

if miss_nonzero.empty:
    print("No missing values in any feature column.")
else:
    print(f"Features with missing values: {len(miss_nonzero)} / {len(feat_cols)}")
    print(f"Max missing rate: {miss_nonzero.iloc[0]:.4%}")
    display(miss_nonzero.head(10).to_frame("missing_rate"))

In [ ]:
# 3c. Class distribution — 8-class cohort balance per phase
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), sharey=False)
for ax, phase in zip(axes, sorted(df_all.phase.unique())):
    sub = df_all[df_all.phase == phase]
    counts = sub.groupby("cohort").size().sort_values(ascending=False)
    colors = [COHORT_COLORS.get(c, "#aaa") for c in counts.index]
    counts.plot.bar(ax=ax, color=colors, edgecolor="white", width=0.7)
    ax.set_title(phase.replace("_", " ").title(), fontsize=9)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.set_ylabel("Windows" if phase == sorted(df_all.phase.unique())[0] else "")

fig.suptitle("Window Counts per Cohort and Phase (all configs combined)", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# 3d. Top discriminative features (Kruskal-Wallis) — full_gait 5s/50%
sub = df_all[(df_all.phase == "full_gait") & (df_all.win_s == 5.0) & (df_all.overlap == 50)]

kw_results = []
for c in feat_cols:
    groups = [sub[sub.label == lbl][c].dropna().values for lbl in ["healthy", "neuro", "ortho"]]
    if all(len(g) > 3 for g in groups):
        H, p = kruskal(*groups)
        kw_results.append({"feature": c, "H": H, "p": p})

kw_df = pd.DataFrame(kw_results).sort_values("H", ascending=False)
print(f"Features significant at p < 0.05: {(kw_df.p < 0.05).sum()} / {len(kw_df)}")
display(kw_df.head(15).reset_index(drop=True))

In [ ]:
# 3e. KDE plots of top 6 discriminative features
top6 = kw_df.head(6)

fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, (_, row) in zip(axes.flatten(), top6.iterrows()):
    feat = row["feature"]
    for lbl, color in LABEL_COLORS.items():
        vals = sub[sub.label == lbl][feat].dropna()
        if len(vals) > 5:
            vals.plot.kde(ax=ax, label=lbl, color=color, linewidth=1.5)
    ax.set_title(f"{feat}\nH={row['H']:.1f}, p={row['p']:.1e}", fontsize=7.5)
    ax.legend(fontsize=6)
    ax.tick_params(labelsize=6)
    ax.set_xlabel("")

fig.suptitle("Top 6 Discriminative Features — Class-wise KDE (full_gait 5s/50%)",
             fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()

<a id="4-3class"></a>
## 4. ML Retraining — 3-Class Results

We retrained SVM, XGBoost, and Random Forest on the corrected features using **5-fold StratifiedGroupKFold** (grouped by `subject_id` to prevent data leakage).

Models tested across multiple phase/window/overlap configurations. Results were pre-computed in `analysis/train_new_features.py`.

In [ ]:
# 4a. Load and display 3-class results
df_3c = pd.read_csv(ML_DIR / "3class_results.csv")
df_3c["bacc_pct"] = (df_3c["bacc"] * 100).round(1)
df_3c["f1_pct"] = (df_3c["f1"] * 100).round(1)

# Best result per phase
best_per_phase = df_3c.loc[df_3c.groupby("phase")["bacc"].idxmax()]
best_per_phase["delta_vs_old"] = best_per_phase["bacc_pct"] - 71.6  # old best was 71.6%

print("=== 3-Class: Best Model per Phase ===")
print(f"Old best (buggy features): 71.6% BAcc\n")
display(best_per_phase[["phase", "win_s", "overlap_pct", "model", "bacc_pct", "f1_pct", "delta_vs_old"]]
        .rename(columns={"bacc_pct": "BAcc %", "f1_pct": "F1 %", "delta_vs_old": "Δ vs Old"})
        .reset_index(drop=True))

In [ ]:
# 4b. Bar chart of 3-class results by phase
fig, ax = plt.subplots(figsize=(10, 4.5))
phases = sorted(df_3c.phase.unique())
models = ["SVM", "XGBoost", "RF"]
colors = {"SVM": "#3498DB", "XGBoost": "#E74C3C", "RF": "#2ECC71"}
x = np.arange(len(phases))
width = 0.22

for i, model in enumerate(models):
    best_vals = []
    for p in phases:
        sub = df_3c[(df_3c.phase == p) & (df_3c.model == model)]
        best_vals.append(sub["bacc_pct"].max())
    ax.bar(x + i * width, best_vals, width, label=model, color=colors[model], alpha=0.85)

ax.axhline(71.6, color="grey", linestyle="--", linewidth=1, label="Old best (71.6%)")
ax.axhline(33.3, color="lightgrey", linestyle=":", linewidth=0.8, label="Chance (33.3%)")
ax.set_xticks(x + width)
ax.set_xticklabels([p.replace("_", " ") for p in phases], fontsize=8)
ax.set_ylabel("Balanced Accuracy (%)")
ax.set_title("3-Class Classification — Best BAcc per Phase (corrected features)")
ax.legend(fontsize=7)
ax.set_ylim(30, 85)
plt.tight_layout()
plt.show()

<a id="5-8class"></a>
## 5. ML Retraining — 8-Class (Subtype-Level)

This is a **new experiment** — classifying all 8 cohorts individually (HS, PD, CVA, RIL, CIPN, KOA, HOA, ACL). This was impossible with the old features because the `cohort` label was missing.

In [ ]:
# 5a. Load and display 8-class results
df_8c = pd.read_csv(ML_DIR / "8class_results.csv")
df_8c["bacc_pct"] = (df_8c["bacc"] * 100).round(1)
df_8c["f1_pct"] = (df_8c["f1"] * 100).round(1)

print("=== 8-Class Results (all phase/model combos) ===")
print(f"Chance level: {100/8:.1f}%\n")
display(df_8c[["phase", "model", "bacc_pct", "f1_pct"]]
        .rename(columns={"bacc_pct": "BAcc %", "f1_pct": "F1 %"})
        .sort_values("BAcc %", ascending=False)
        .reset_index(drop=True))

best8 = df_8c.loc[df_8c.bacc.idxmax()]
print(f"\n>>> Best 8-class: {best8['model']} on {best8['phase']} → "
      f"BAcc = {best8['bacc_pct']}% (vs {100/8:.1f}% chance = {best8['bacc_pct']/(100/8):.1f}x above chance)")

<a id="6-ablation"></a>
## 6. Sensor Ablation

Which sensors are most important? We tested all individual sensors and key combinations on the best 3-class config (full_gait, 5s, 50%).

In [ ]:
# 6a. Sensor ablation results
df_abl = pd.read_csv(ML_DIR / "sensor_ablation_results.csv")

pivot = (df_abl.pivot_table(index="sensor_set", columns="model", values="bacc")
         .sort_values("XGBoost", ascending=False) * 100).round(1)

print("=== Sensor Ablation — BAcc % (full_gait, 5s, 50%, 3-class) ===")
display(pivot)

# Highlight HE finding
he_xgb = pivot.loc["HE only", "XGBoost"]
all_xgb = pivot.loc["All (HE+LB+LF+RF)", "XGBoost"]
print(f"\n>>> Surprise: HE only ({he_xgb}%) retains {he_xgb/all_xgb*100:.1f}% of All-sensor ({all_xgb}%) performance.")
print(">>> A single head-mounted IMU may be sufficient for 3-class gait screening.")

In [ ]:
# 6b. Sensor ablation bar chart
fig, ax = plt.subplots(figsize=(10, 4.5))
model_colors = {"SVM": "#3498DB", "XGBoost": "#E74C3C", "RF": "#2ECC71"}
pivot.plot.bar(ax=ax, color=[model_colors[m] for m in pivot.columns], alpha=0.85, edgecolor="white")
ax.axhline(33.3, color="lightgrey", linestyle=":", linewidth=0.8, label="Chance")
ax.set_ylabel("Balanced Accuracy (%)")
ax.set_title("Sensor Ablation — 3-Class (full_gait 5s/50%)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", fontsize=7)
ax.set_ylim(30, 85)
ax.legend(title="Model", fontsize=7)
plt.tight_layout()
plt.show()

<a id="7-expanded"></a>
## 7. Expanded Feature Experiment

We tested whether adding **derived proxy features** from existing statistics would improve results:
- `energy` = rms²
- `dc_ratio` = |mean| / rms
- `rel_var` = std / rms
- `spec_complexity` = spec_centroid × std
- `norm_spec_power` = spec_power / energy

This expanded the feature set from 216 → 396.

In [ ]:
# 7a. Load expanded feature comparison
df_exp = pd.read_csv(ML_DIR / "expanded_3class_results.csv")
df_exp["bacc_pct"] = (df_exp["bacc"] * 100).round(1)
df_exp["f1_pct"] = (df_exp["f1"] * 100).round(1)

print("=== Original (216) vs Expanded (396) Features ===")
display(df_exp[["feature_set", "model", "bacc_pct", "f1_pct", "n_features"]]
        .rename(columns={"bacc_pct": "BAcc %", "f1_pct": "F1 %"})
        .reset_index(drop=True))

print("\n>>> Conclusion: Expanded features do NOT improve results.")
print(">>> Proxy features (e.g., rms² ≈ energy) are correlated with existing stats.")
print(">>> To get genuinely new features, we need access to raw IMU signals for")
print("    kurtosis, skewness, zero-crossing rate, wavelet coefficients, etc.")

<a id="8-fixes"></a>
## 8. Professor Feedback Fixes — Figure 7 & Table II

### Feedback item 5: Figure 7
- **Problem:** VGA is an ordinal scale (0–4). OLS regression line was inappropriate.
- **Fix:** Removed regression line. Added Spearman ρ annotation + per-VGA-category boxplots.

### Feedback item 6: Table II (Cohen's d)
- **Problem:** All d values were unsigned (positive).
- **Fix:** Signed convention: d = (pathological − healthy) / pooled SD → all negative.

In [ ]:
# 8a. Display corrected Table II
table2 = pd.read_csv(RESULTS / "artifacts" / "table2_corrected.csv")
print("=== Table II (Corrected) — Signed Cohen's d ===")
print("Convention: d = (pathological − healthy) / pooled SD")
print("Negative values = pathological group has LOWER |AI| than healthy\n")
display(table2)

In [ ]:
# 8b. Display corrected Figure 7
from IPython.display import Image as IPImage, display as ipdisplay

fig7_path = RESULTS / "figures" / "step07_corr_vga_stride_absAI_fixed.png"
if fig7_path.exists():
    print("=== Corrected Figure 7 ===")
    print("Panel A: Scatter (no regression line) + Spearman ρ")
    print("Panel B: Per-VGA-category boxplots\n")
    ipdisplay(IPImage(filename=str(fig7_path), width=700))
else:
    print(f"Figure not found at: {fig7_path}")

<a id="9-summary"></a>
## 9. Summary & Key Insights

### What we fixed
| # | Professor's Feedback | Action Taken |
|---|---------------------|--------------|
| 1 | "robust biomarkers" too strong | → "potential indicators", "suggests" |
| 2 | Streamline narrative | Abstract rewritten with 2 explicit contributions |
| 3 | ML improvement is small | AUC 0.716 labeled "modest"; clinical insight emphasized |
| 4 | Sensor ablation needs key-takeaway | Added: "Foot-only retains 93% BAcc" |
| 5 | Fig 7: regression line on ordinal VGA | Removed; added Spearman ρ + boxplots |
| 6 | Table II: RIL should be d = −0.87 | All Cohen's d now negative (signed convention) |

### Key Insights

1. **Feature extraction > model tuning.** Fixing the missing Acc channel + window-level data → +7.6% BAcc improvement. More impactful than any hyperparameter search.

2. **Head sensor is underrated.** HE alone (74.0%) ≈ all 4 sensors (75.7%). A single head-mounted IMU could be a practical screening device.

3. **Subtype discrimination is feasible.** 8-class BAcc = 41.5% (3.3× chance). IMU features carry cohort-specific signatures.

4. **Straight-line walking = main signal.** Pre-U-turn phase is statistically significant (p = 0.026). ML performs best on post_uturn (79.2%) — likely cleaner signal.

5. **VGA is a coarse proxy.** ρ = −0.206 → VGA explains only ~4% of IMU asymmetry variance. IMU captures what visual rating cannot.

### Next Steps (Week 4)
- Access raw IMU signals → compute kurtosis, ZCR, wavelet energy
- 8-class with SMOTE / class-weighted loss for RIL/ACL imbalance
- Phase-feature fusion (pre + post uturn)
- Discuss head-sensor-only finding with professor